# 02 - RAG Pipeline & RAGAS Evaluation (AWS Bedrock)

Ce notebook :
1. Construit le pipeline RAG avec différentes configurations (Claude + Titan sur Bedrock)
2. Exécute l'évaluation RAGAS sur chaque configuration
3. Sauvegarde les résultats pour analyse

In [ ]:
import sys
sys.path.insert(0, '..')

import boto3

from src.config import EXPERIMENT_CONFIGS, AWS_REGION
from src.rag_pipeline import RAGPipeline
from src.evaluation import run_evaluation, run_all_experiments
from src.data_loader import load_corpus_from_texts, load_testset

# Vérifier la connexion AWS
sts = boto3.client('sts', region_name=AWS_REGION)
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account']}")
print(f"Region: {AWS_REGION}")
print(f"Role: {identity['Arn'].split('/')[-1]}")
print(f"\nNumber of experiment configs: {len(EXPERIMENT_CONFIGS)}")

## 1. Charger les données

In [ ]:
documents = load_corpus_from_texts('../data/corpus')
questions, ground_truths = load_testset('../data/testset/testset.json')

print(f"\nCorpus: {len(documents)} documents")
print(f"Testset: {len(questions)} questions")

## 2. Test rapide avec une seule configuration

Avant de lancer tous les benchmarks, testons avec la config baseline + Claude Sonnet sur Bedrock.

In [ ]:
config = EXPERIMENT_CONFIGS[1]  # baseline_medium_chunks
print(f"Testing: {config}")

pipeline = RAGPipeline(config)
pipeline.load_documents(documents)
pipeline.chunk_documents()
pipeline.build_index()

# Test une seule question
result = pipeline.query(questions[0])
print(f"\nQuestion: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Contexts retrieved: {len(result['contexts'])}")
print(f"Latency: {result['latency']:.2f}s")

pipeline.cleanup()

## 3. Exécuter tous les benchmarks

⚠️ Cette cellule peut prendre 20-40 minutes selon le nombre de configs.

Modèles utilisés :
- **LLM générateur** : Claude Sonnet 4, Claude Haiku 3, Claude Haiku 4.5
- **Embeddings** : Amazon Titan Embed Text V2 (via Bedrock) + all-MiniLM-L6-v2 (local)
- **LLM juge RAGAS** : Claude Sonnet 4 (via Bedrock)

In [ ]:
# Option 1: Toutes les configs (complet)
# configs_to_test = EXPERIMENT_CONFIGS

# Option 2: Sous-ensemble pour test rapide
configs_to_test = EXPERIMENT_CONFIGS[:4]

print(f"Running {len(configs_to_test)} configurations...")
for i, cfg in enumerate(configs_to_test):
    print(f"  {i+1}. {cfg.name} (LLM: {cfg.llm.model_name})")

In [ ]:
results_df = run_all_experiments(
    configs=configs_to_test,
    documents=documents,
    test_questions=questions,
    ground_truths=ground_truths,
    output_dir='../data/results',
    judge_model='anthropic.claude-sonnet-4-20250514-v1:0',
)

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
results_df[['config_name', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'avg_latency']]

## 4. Aperçu rapide des résultats

In [ ]:
import pandas as pd

metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
results_df['avg_score'] = results_df[metrics].mean(axis=1)

print("\nClassement par score moyen RAGAS :")
ranked = results_df.sort_values('avg_score', ascending=False)[['config_name', 'avg_score', 'avg_latency']]
for i, (_, row) in enumerate(ranked.iterrows()):
    print(f"  {i+1}. {row['config_name']:<25} Score: {row['avg_score']:.4f}  Latency: {row['avg_latency']:.2f}s")